In [12]:
import pandas as pd
import numpy as np

In [16]:
df=pd.read_csv("/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/apple/merged_apple_funadmentals_price_macro.csv")

In [17]:
df

,quarter,period_end,Price,revenue,net_income,ratio assets/libailities,shareholders_equity,gdp_growth,interest_rate
0,Q1 2010,2009-12-26,6.407192,15683000000,3378000000,2.969820,35768000000,0.11,0.120000
1,Q2 2010,2010-03-27,7.127073,13499000000,3074000000,3.221921,39348000000,1.75,0.133333
2,Q3 2010,2010-06-26,7.634115,15700000000,3253000000,2.994587,43111000000,2.91,0.193333
3,Q4 2010,2010-09-25,8.590257,20343000000,4308000000,2.744706,47791000000,3.34,0.186667
4,Q1 2011,2010-12-25,9.775751,26741000000,6004000000,2.704265,54666000000,2.78,0.186667
...,...,...,...,...,...,...,...,...,...
58,Q3 2024,2024-06-29,210.863424,85777000000,21448000000,1.251820,66708000000,3.04,5.330000
59,Q4 2024,2024-09-28,228.456772,94930000000,14736000000,1.184885,56950000000,2.72,5.263333
60,Q1 2025,2024-12-28,248.049447,124300000000,36330000000,1.240719,66758000000,2.53,4.650000
61,Q2 2025,2025-03-29,219.273273,95359000000,24780000000,1.252597,66796000000,1.99,4.330000


In [18]:
# --------- 1) Feature engineering: make leak-safe lags ---------


def make_lag_features(
    df,
    date_col="period_end",
    target_col="Price",
    exog_cols=("revenue","net_income","ratio assets/libailities",
               "shareholders_equity","gdp_growth","interest_rate"),
    lags=(1,2,4),
    horizon=1,  # predict next quarter by default
):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)

    # Target: next-quarter return (you can switch to .shift(-horizon) on levels if you prefer)
    df["ret"] = df[target_col].pct_change()
    y = df["ret"].shift(-horizon)  # what we want to predict

    # Create lagged predictors (only past info)
    cols_to_lag = ["ret"] + list(exog_cols)
    for c in cols_to_lag:
        for L in lags:
            df[f"{c}_lag{L}"] = df[c].shift(L)

    # Optional: time markers (seasonality)
    df["year"] = df.index.year
    df["quarter_num"] = df.index.quarter
    # cyclic encoding for quarter (keeps it concise & model-friendly)
    df["q_sin"] = np.sin(2*np.pi*df["quarter_num"]/4)
    df["q_cos"] = np.cos(2*np.pi*df["quarter_num"]/4)

    # Final feature set
    feature_cols = [c for c in df.columns if c.endswith(tuple(f"_lag{L}" for L in lags))] + ["q_sin","q_cos"]
    X = df[feature_cols]

    # Drop rows with NaNs from lagging and target shift
    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask], y[mask]
    return X, y


In [19]:
# --------- 2) Train / test split (time-based) + XGBoost ---------
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Build dataset
X, y = make_lag_features(df)

# Simple chronological split (last 20% as test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.4f}   R²: {r2:.3f}")

# Optional: feature importance (quick peek)
imp = sorted(zip(model.feature_names_in_, model.feature_importances_), key=lambda x: -x[1])[:12]
for name, val in imp:
    print(f"{name:30s} {val:.3f}")


RMSE: 0.0444   R²: -1.668
interest_rate_lag4             0.161
gdp_growth_lag2                0.110
ret_lag2                       0.063
ratio assets/libailities_lag2  0.061
interest_rate_lag2             0.052
net_income_lag4                0.051
interest_rate_lag1             0.048
shareholders_equity_lag1       0.045
ratio assets/libailities_lag1  0.043
ret_lag4                       0.039
gdp_growth_lag1                0.036
shareholders_equity_lag2       0.036
